# Pexels API (Week 4)

This notebook calls the [Pexels Search API](https://www.pexels.com/api/documentation/#photos-search) using **`PEXELS_API_KEY`** from `Week 4/.env`.

**Setup:** start Jupyter from the repo root or from `Week 4/` so the notebook can find `.env`. If you use a venv:

```bash
cd "Week 4"
.venv/bin/python -m pip install requests jupyter ipykernel -q
.venv/bin/python -m ipykernel install --user --name=hcde530-week4
```

Or install `requests` (and optionally `jupyter`) into whatever kernel you select.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import requests

PEXELS_SEARCH = "https://api.pexels.com/v1/search"


def find_env_dir() -> Path:
    """Locate the folder that contains `.env` (this notebook lives in Week 4)."""
    d = Path.cwd().resolve()
    for candidate in (d, d / "Week 4", d.parent / "Week 4"):
        env_file = candidate / ".env"
        if env_file.is_file():
            return candidate
    return d


def load_pexels_key(env_path: Path) -> str | None:
    """Same rules as `export_pexels_to_csv.py`: utf-8-sig, KEY=value, or one raw line."""
    if not env_path.is_file():
        return None
    raw = env_path.read_text(encoding="utf-8-sig").strip()
    if not raw:
        return None
    first = raw.splitlines()[0].strip()
    if "=" not in first:
        return raw.strip()
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k == "PEXELS_API_KEY":
            return v
    return None


HERE = find_env_dir()
ENV_FILE = HERE / ".env"
API_KEY = load_pexels_key(ENV_FILE) or os.environ.get("PEXELS_API_KEY")

if not API_KEY:
    raise SystemExit(
        f"No PEXELS_API_KEY found. Add it to {ENV_FILE} or export PEXELS_API_KEY. "
        "See https://www.pexels.com/api/"
    )

print("Loaded API key from:", ENV_FILE)
print("Key prefix:", API_KEY[:6] + "…" if len(API_KEY) > 6 else "(short)")

## Search photos

`GET https://api.pexels.com/v1/search` with query params `query`, `per_page`, `page` and header `Authorization: <your-api-key>`.

In [ ]:
def search_pexels(query: str, *, per_page: int = 5, page: int = 1) -> dict:
    resp = requests.get(
        PEXELS_SEARCH,
        params={"query": query, "per_page": per_page, "page": page},
        headers={"Authorization": API_KEY},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()


data = search_pexels("pharmacy stock photo", per_page=5, page=1)
print("page:", data.get("page"), "per_page:", data.get("per_page"), "total_results:", data.get("total_results"))
photos = data.get("photos") or []
print("photos in this page:", len(photos))
for p in photos:
    pid = p.get("id")
    url = (p.get("src") or {}).get("medium") or (p.get("src") or {}).get("small")
    print(f"  id={pid} photographer={p.get('photographer')!r} url={url}")

## Preview one image (optional)

Uses `IPython.display.Image` to show a thumbnail URL from the last search (no download required).

In [ ]:
from IPython.display import Image, display

if photos:
    first = photos[0]
    thumb = (first.get("src") or {}).get("medium") or (first.get("src") or {}).get("small")
    if thumb:
        display(Image(url=thumb, width=400))
        print("Pexels page:", first.get("url"))
else:
    print("No photos to display.")